In [ ]:
# !pip install -q pydantic pydantic-ai openai python-dotenv duckduckgo-search rich

## Deep dive into agents! 

In this tutorial, we'll explore why structure matters in AI agents and how PydanticAI 
provides a robust framework for building reliable, predictable agents.

Think of traditional agents as freestyle conversations - they work, but can be unpredictable.
Structured agents are like having a well-designed form that guides the conversation 
toward specific, reliable outcomes.

Learning Objectives:
- Understand the limitations of unstructured agent interactions
- Learn how PydanticAI enforces structure through schemas
- Build a simple structured agent that returns predictable results


In [ ]:
import os
from typing import List, Dict, Any
from pydantic import BaseModel, Field
from pydantic_ai import Agent, RunContext
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

### Understanding the Problem with Unstructured Agents

Let's start by seeing what happens when we use a basic, unstructured approach.
This will help us understand WHY we need structure.


In [ ]:
# OpenRouter setup
from getpass import getpass
import os

OPENROUTER_API_KEY = getpass("Enter your OpenRouter API key: ")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
MODEL = "openai/gpt-4o-mini"

from openai import OpenAI
client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_BASE_URL,
)

prompt = "Analyze this product and tell me if I should buy it: iPhone 15 Pro"

response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": prompt}]
)

# The problem: we get unstructured text that's hard to parse
response.choices[0].message.content

### Defining Structure with Pydantic Models

Now let's see how we can define exactly what we want our agent to return.
This is the foundation of structured agents - defining the shape of our data.



In [ ]:
class ProductAnalysis(BaseModel):
    """
    This model defines exactly what our agent should return.
    Notice how we're being very specific about data types and descriptions.
    """
    product_name: str = Field(description="The name of the product being analyzed")
    recommendation: str = Field(description="Either 'buy', 'don't buy', or 'maybe'")
    confidence_score: float = Field(description="Confidence from 0.0 to 1.0", ge=0.0, le=1.0)
    key_pros: List[str] = Field(description="Top 3 advantages of this product")
    key_cons: List[str] = Field(description="Top 3 disadvantages of this product")
    reasoning: str = Field(description="Brief explanation of the recommendation")

### Creating Our First Structured Agent

Now we'll combine our schema with PydanticAI to create an agent that 
ALWAYS returns data in our specified format.



In [ ]:
product_analyzer = Agent(
    openrouter_model,
    output_type=ProductAnalysis,
    system_prompt="""
    You are a product analysis expert. Analyze products objectively and provide 
    structured recommendations based on features, price, reviews, and market position.
    
    Always be honest about limitations and provide balanced viewpoints.
    Your confidence score should reflect the strength of available evidence.
    """
)

### Adding Intelligence to Our Agent

Let's enhance our agent with some basic reasoning capabilities.
We'll add a function that helps our agent access current information.



In [ ]:
@product_analyzer.tool
async def get_product_info(ctx: RunContext[None], product_name: str) -> str:
    """
    Simulated tool that would normally fetch real product data.
    In a real implementation, this might call APIs, scrape websites, etc.
    """
    # Simulated product database - in reality, this would be dynamic
    product_db = {
        "iPhone 15 Pro": {
            "price": "$999",
            "features": ["48MP camera", "A17 Pro chip", "Titanium build", "USB-C"],
            "reviews": "4.5/5 stars average",
            "availability": "In stock"
        },
        "Samsung Galaxy S24": {
            "price": "$899", 
            "features": ["200MP camera", "Snapdragon 8 Gen 3", "7 years updates"],
            "reviews": "4.4/5 stars average",
            "availability": "In stock"
        }
    }
    
    product_info = product_db.get(product_name, {"error": "Product not found"})
    return f"Product info for {product_name}: {product_info}"


### Running Our Structured Agent

Now let's see our structured agent in action and compare it to the unstructured approach.


In [ ]:

# Run our structured agent
result = await product_analyzer.run("Should I buy the iPhone 15 Pro?")

In [ ]:
print("Structured output:")
print(f"Product: {result.output.product_name}")
print(f"Recommendation: {result.output.recommendation}")
print(f"Confidence: {result.output.confidence_score:.2f}")
print(f"Pros: {', '.join(result.output.key_pros)}")
print(f"Cons: {', '.join(result.output.key_cons)}")
print(f"Reasoning: {result.output.reasoning}")